In [2]:
import pandas as pd
import numpy as np
import re
import os

# 📁 Định nghĩa đường dẫn gốc
# Cách 1: Dùng đường dẫn tuyệt đối (đơn giản nhất)
BASE_PATH = r"C:\Users\ThaoMuy\Downloads\Recommend_System"

# Cách 2: Tự động phát hiện từ vị trí notebook (uncomment nếu muốn dùng)
# import pathlib
# NOTEBOOK_FILE = pathlib.Path(__file__ if '__file__' in globals() else 'pre_process.ipynb')
# BASE_PATH = NOTEBOOK_FILE.parent.parent.parent.parent  # backend/data/recipes/ -> Recommend_System/

# 📂 Đường dẫn thư mục con
DATA_RECIPES = os.path.join(BASE_PATH, "backend", "data", "recipes")
DATA_USERS = os.path.join(BASE_PATH, "backend", "data", "users")

print(f"📁 BASE_PATH: {BASE_PATH}")
print(f"📂 DATA_RECIPES: {DATA_RECIPES}")
print(f"📂 DATA_USERS: {DATA_USERS}")

📁 BASE_PATH: C:\Users\ThaoMuy\Downloads\Recommend_System
📂 DATA_RECIPES: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes
📂 DATA_USERS: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\users


In [12]:
# 📂 File gốc
csv_path = os.path.join(DATA_RECIPES, "full_data_1.csv")
output_path = csv_path.replace(".csv", "_clean.csv")

# 1️⃣ Đọc file gốc an toàn
df = pd.read_csv(csv_path)

print(f"📂 Đã đọc file: {csv_path}")
print(f"📊 Tổng số dòng ban đầu: {len(df)} | Số cột: {len(df.columns)}")

# 2️⃣ Làm sạch dữ liệu
required_cols = ["title", "description", "url", "ingredients", "instructions", "servings"]
required_cols = [c for c in required_cols if c in df.columns]

df_clean = df.dropna(subset=required_cols)
df_clean = df_clean[
    (df_clean["title"].astype(str).str.strip() != "") &
    (df_clean["url"].astype(str).str.strip() != "")
]

print(f"🧹 Đã xoá {len(df) - len(df_clean)} dòng thiếu dữ liệu quan trọng.")
print(f"✅ Còn lại {len(df_clean)} dòng hợp lệ sau khi lọc.")

# 3️⃣ Xoá trùng title
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title"], keep="first")
print(f"🧩 Đã xoá {before_dup - len(df_clean)} dòng trùng title.")
print(f"📈 Còn lại {len(df_clean)} dòng sau khi xoá trùng.")

# 4️⃣ Xoá cột 'source_file' nếu có
if "source_file" in df_clean.columns:
    df_clean = df_clean.drop(columns=["source_file"])
    print("🗑️ Đã xoá cột 'source_file'.")

# 5️⃣ Sắp xếp lại thứ tự cột
new_order = [
    "title", "url", "description",
    "prep_time", "cook_time", "total_time",
    "rating_value", "rating_count", "review_count"
]
remaining_cols = [c for c in df_clean.columns if c not in new_order]
df_clean = df_clean[new_order + remaining_cols]

print("📑 Thứ tự 10 cột đầu tiên:", df_clean.columns[:10].tolist())

# 6️⃣ Lưu file kết quả
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"🎉 File đã clean và lưu tại: {output_path}")

📂 Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_1.csv
📊 Tổng số dòng ban đầu: 40638 | Số cột: 23
🧹 Đã xoá 95 dòng thiếu dữ liệu quan trọng.
✅ Còn lại 40543 dòng hợp lệ sau khi lọc.
🧩 Đã xoá 99 dòng trùng title.
📈 Còn lại 40444 dòng sau khi xoá trùng.
📑 Thứ tự 10 cột đầu tiên: ['title', 'url', 'description', 'prep_time', 'cook_time', 'total_time', 'rating_value', 'rating_count', 'review_count', 'servings']
🎉 File đã clean và lưu tại: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_1_clean.csv


In [5]:
# 📂 Đường dẫn tới file đã merge
csv_path = os.path.join(DATA_RECIPES, "all_recipes.csv")  

# Đọc file
df = pd.read_csv(csv_path)

# Hiển thị thông tin cơ bản
print("✅ Đã đọc dữ liệu thành công!")
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
display(df.head(3))

✅ Đã đọc dữ liệu thành công!
Số dòng: 40684
Số cột: 23


,title,description,url,prep_time,cook_time,total_time,servings,ingredients,instructions,image_url,...,rating_value,rating_count,review_count,yield,protein,fat,carbohydrate,fiber,sugar,sodium
0,\nCaesar Scalloped Potatoes\n,These Caesar scalloped potatoes really taste l...,https://www.allrecipes.com/caesar-scalloped-po...,15m,95m,120m,12,"[""¼ cup butter"", ""1 medium onion, chopped (1 c...","[""1) Gather all ingredients. Preheat the oven ...",https://www.allrecipes.com/thmb/HjKdZEAOiGXvd1...,...,3.8,5.0,4.0,NaN,7 g,7 g,38 g,4 g,3 g,364 mg
1,\nChili Crisp Marinated Cheese\n,Chili crisp marinated cheese is an easy appeti...,https://www.allrecipes.com/chili-crisp-marinat...,15m,NaN,15m,12,"[""12 ounces cream cheese, sliced"", ""12 ounces ...","[""1) In a 8x8 dish, line the sliced cheeses in...",https://www.allrecipes.com/thmb/jskEZczArV2tBB...,...,NaN,NaN,0.0,NaN,16 g,32 g,7 g,0 g,6 g,443 mg
2,\nNo-Bake Millionaire’s Shortbread\n,This no-bake millionaire's shortbread has a bu...,https://www.allrecipes.com/no-bake-millionaire...,25m,10m,285m,24,"[""10 ounces shortbread cookies (about 2 2/3 cu...","[""1) Gather all ingredients. Grease a 9x9-inch...",https://www.allrecipes.com/thmb/X-Km8Jo7nZqp4h...,...,3.0,2.0,1.0,24 bar cookies,3 g,16 g,30 g,1 g,23 g,156 mg


In [6]:
# ===============================================

# 1️⃣ Đọc file
src = os.path.join(DATA_RECIPES, "all_recipes.csv")
df = pd.read_csv(src, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {src} ({len(df)} dòng)")

if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong CSV!")

# 2️⃣ Danh sách cần gán 'World'
drop_list = {
    "Dessert", "Baking", "Dog Food", "Universal", "Authentic", 
    "Copycat", "Inspired", "World"
}

# 3️⃣ Hàm chuẩn hoá text
def normalize_text(s):
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())

# 4️⃣ Từ điển nhóm chuẩn (được gom nhóm theo danh sách 117 cuisine của bạn)
groups = {
    "Vietnamese": ["vietnamese"],
    "Chinese": ["chinese", "sichuan"],
    "Japanese": ["japanese"],
    "Korean": ["korean"],
    "Thai": ["thai"],
    "Indian": ["indian", "bangladeshi", "pakistani", "sri lankan"],
    "Italian": ["italian", "sicilian"],
    "French": ["french", "french canadian"],
    "British": ["british", "english", "uk", "scottish", "irish", "welsh"],
    "American": [
        "american", "u.s.", "north american", "native american", "southwestern",
        "southern", "hawaiian", "new england", "pennsylvania dutch", "amish",
        "tex mex", "tex-mex", "cajun", "creole"
    ],
    "Mexican": ["mexican", "latin", "latin american", "south american", "argentine", "brazilian", "venezuelan"],
    "Mediterranean": [
        "greek", "lebanese", "turkish", "israeli", "persian", "moroccan", 
        "egyptian", "tunisian", "middle eastern", "north african"
    ],
    "European": [
        "european", "austrian", "german", "swiss", "hungarian", "polish", "spanish", 
        "portuguese", "dutch", "danish", "norwegian", "finnish", "swedish", 
        "scandinavian", "russian", "ukrainian", "eastern european"
    ],
    "Australian": ["australian", "new zealand", "oceanic"],
    "Asian": [
        "asian", "east and southeast asian", "south and central asian",
        "asian inspired", "asian fusion", "filipino", "malaysian", "indonesian", 
        "singaporean"
    ],
    "African": ["african", "ethiopian", "west african", "south african", "east african"],
    "Caribbean": ["caribbean", "jamaican", "cuban", "puerto rican", "trinidad", "salvadoran", "colombian", "chilean", "peruvian"],
    "Jewish": ["jewish", "kosher"],
    "Fusion": ["fusion", "modern", "inspired"]
}

# 5️⃣ Chuẩn hoá từng recipe, giữ nguyên multi-cuisine
def normalize_cuisine_field(cuisine_field):
    if pd.isna(cuisine_field) or str(cuisine_field).strip() == "":
        return "World"

    cuisines = [c.strip() for c in str(cuisine_field).split(",") if c.strip()]
    normalized = []

    for c in cuisines:
        cname = normalize_text(c)
        # Nếu nằm trong drop_list hoặc dạng "Inspired" → World
        if any(dl.lower() == cname for dl in drop_list):
            normalized.append("World")
            continue

        matched = None
        for group, kws in groups.items():
            for kw in kws:
                if kw in cname:
                    matched = group
                    break
            if matched:
                break
        normalized.append(matched or "World")

    # Loại trùng, giữ thứ tự
    seen, result = set(), []
    for item in normalized:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return ", ".join(result)

# 6️⃣ Áp dụng
df["recipe_cuisine"] = df["recipe_cuisine"].apply(normalize_cuisine_field)

# 7️⃣ Lưu file mới
out_path = src.replace(".csv", "_normalized.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Đã chuẩn hoá cột 'recipe_cuisine' (fallback = 'World')")
print(f"💾 File lưu tại: {out_path}")

# 8️⃣ Thống kê tần suất
cuisine_counts = {}
for cell in df["recipe_cuisine"]:
    for c in [x.strip() for x in str(cell).split(",") if x.strip()]:
        cuisine_counts[c] = cuisine_counts.get(c, 0) + 1

print("\n📊 Top 20 cuisine phổ biến nhất:")
for c, cnt in sorted(cuisine_counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{c:15} → {cnt:5}")


✅ Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes.csv (40684 dòng)

✅ Đã chuẩn hoá cột 'recipe_cuisine' (fallback = 'World')
💾 File lưu tại: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized.csv

📊 Top 20 cuisine phổ biến nhất:
American        → 20347
World           → 10765
Italian         →  2239
Mexican         →  2038
Asian           →  1522
European        →  1057
Mediterranean   →   862
French          →   567
British         →   555
Indian          →   528
Vietnamese      →   444
Korean          →   390
Chinese         →   374
Caribbean       →   340
Thai            →   232
Japanese        →   227
Fusion          →   207
Jewish          →   130
African         →   123
Australian      →    61


In [7]:

# 1️⃣ Đọc file gốc
input_path = os.path.join(DATA_RECIPES, "all_recipes_normalized.csv")
output_path = input_path.replace(".csv", "_time.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path}")
print(f"📦 Số dòng: {len(df)} | Số cột: {len(df.columns)}")
print(f"🔍 Số lượng giá trị thiếu:\n{df[["prep_time", "cook_time", "total_time"]].isnull().sum()}")
# 2️⃣ Chuyển về phút
def to_minutes(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = re.sub(r"[^0-9hms ]", "", s)
    if "h" in s:
        h = re.search(r"(\d+)\s*h", s)
        m = re.search(r"(\d+)\s*m", s)
        hours = int(h.group(1)) if h else 0
        mins = int(m.group(1)) if m else 0
        return hours * 60 + mins
    m = re.search(r"(\d+)", s)
    return int(m.group(1)) if m else np.nan

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(to_minutes)

# 3️⃣ Logic đơn giản
# Nếu có total, gắn cho cook/prep nếu bị NaN
df.loc[df["total_time"].notna() & df["prep_time"].isna(), "prep_time"] = df["total_time"]
df.loc[df["total_time"].notna() & df["cook_time"].isna(), "cook_time"] = df["total_time"]

# Nếu total bị NaN → lấy cook hoặc prep
df.loc[df["total_time"].isna() & df["cook_time"].notna(), "total_time"] = df["cook_time"]
df.loc[df["total_time"].isna() & df["prep_time"].notna(), "total_time"] = df["prep_time"]

# 4️⃣ Format lại dạng 'Xm'
def fmt_m(x):
    return f"{int(x)}m" if pd.notna(x) else ""

for c in ["prep_time", "cook_time", "total_time"]:
    df[c] = df[c].apply(fmt_m)

# 5️⃣ Xuất file kết quả
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã xử lý thời gian và lưu tại:\n👉 {output_path}")
print("\n🔎 5 dòng đầu sau xử lý:")
print(df[["title", "prep_time", "cook_time", "total_time"]].head().to_string(index=False))


✅ Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized.csv
📦 Số dòng: 40684 | Số cột: 23
🔍 Số lượng giá trị thiếu:
prep_time     1572
cook_time     6989
total_time     277
dtype: int64
✅ Đã xử lý thời gian và lưu tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized_time.csv

🔎 5 dòng đầu sau xử lý:
                                title prep_time cook_time total_time
        \nCaesar Scalloped Potatoes\n       15m       95m       120m
     \nChili Crisp Marinated Cheese\n       15m       15m        15m
 \nNo-Bake Millionaire’s Shortbread\n       25m       10m       285m
      4-Ingredient Pepper Pizza Bites        5m       15m        20m
          Air Fryer Green Bean Fries        20m       25m        45m


In [ ]:
import pandas as pd
import numpy as np
import re
import os

# ============================================================
# 🔧 SCRIPT CHUẨN HOÁ ĐƠN VỊ TRONG CỘT "ingredients" & "instructions"
#
# ✔ ounce → gram
# ✔ pound / lb / lbs → gram
# ✔ tbsp / tsp → gram
# ✔ xử lý unicode phân số: ½ ¼ ¾ ⅓ …
# ✔ xử lý "(28 to )" → "(28g)"
# ✔ hỗ trợ dạng "(2-pound)"
#
# ❌ Không hỗ trợ cup → ml/g
# ❌ Không xử lý "per pound"
# ============================================================


# ===== PARSE SỐ AN TOÀN =====
def safe_parse_quantity(q):
    q = q.strip()

    unicode_fractions = {
        "¼": "1/4", "½": "1/2", "¾": "3/4",
        "⅓": "1/3", "⅔": "2/3",
        "⅛": "1/8", "⅜": "3/8", "⅝": "5/8", "⅞": "7/8"
    }
    for uni, frac in unicode_fractions.items():
        q = q.replace(uni, frac)

    if " " in q and "/" in q:
        try:
            whole, frac = q.split()
            return float(whole) + eval(frac)
        except:
            return None

    if "/" in q:
        try:
            num, den = q.split("/")
            return float(num) / float(den)
        except:
            return None

    try:
        return float(q)
    except:
        return None


# ===== MAIN NORMALIZER =====
def normalize_text(s):
    if pd.isna(s):
        return s
    s = str(s)

    # ===== (3–4 ounce) =====
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?)\s*[-–]?\s*(?:to)?\s*(\d+(?:\.\d+)?)\s*[-\s]*(ounce|ounces|oz)\s*\)",
        lambda m: f"({round(float(m.group(1))*28.35)} to {round(float(m.group(2))*28.35)}g)",
        s, flags=re.IGNORECASE
    )

    # ===== (10 ounce) =====
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?)\s*(ounce|ounces|oz)\s*\)",
        lambda m: f"({round(float(m.group(1))*28.35)}g)",
        s, flags=re.IGNORECASE
    )

    # ===== 8 ounces =====
    s = re.sub(
        r"\b(\d+(?:\.\d+)?)\s*(ounce|ounces|oz)\b",
        lambda m: f"{round(float(m.group(1))*28.35)}g",
        s, flags=re.IGNORECASE
    )
    
    # ===== HANDLE "6-ounce" / "8-ounces" / "12-oz" =====
    s = re.sub(
        r"\b(\d+(?:\.\d+)?)\s*[-]?\s*(ounce|ounces|oz)\b",
        lambda m: f"{round(float(m.group(1)) * 28.35)}g",
        s, flags=re.IGNORECASE
    )
    
    # ===== HANDLE (1/2-ounce each) → (14g each) =====
    s = re.sub(
        r"\(\s*([\d¼½¾⅓⅔⅛⅜⅝⅞\/\.]+)\s*[-]?\s*(ounce|ounces|oz)\s+each\s*\)",
        lambda m: f"({round(safe_parse_quantity(m.group(1)) * 28.35)}g each)",
        s,
        flags=re.IGNORECASE
    )



    # ===== 1 20-ounce package =====
    s = re.sub(
        r"(\b\d+(?:\.\d+)?)\s+(\d+(?:\.\d+)?)[- ]?(ounce|oz)\b",
        lambda m: f"{m.group(1)} ({round(float(m.group(2))*28.35)}g)",
        s, flags=re.IGNORECASE
    )
    s = re.sub(
    r"\(\s*(\d+(?:\.\d+)?)\s*[-]?\s*(ounce|ounces|oz)\s*\)",
    lambda m: f"({round(float(m.group(1)) * 28.35)}g)",
    s, flags=re.IGNORECASE
    )

    # ===== TBSP / TSP =====
    conv = {
        "tbsp": 15, "tablespoon": 15, "tablespoons": 15,
        "tsp": 5, "teaspoon": 5, "teaspoons": 5,
    }

    def add_g(m):
        qty_raw = m.group(1)
        unit = m.group(2).lower()
        qty = safe_parse_quantity(qty_raw)
        if qty is None:
            return m.group(0)
        return f"{qty_raw}{unit} ({conv[unit]}g)"

    s = re.sub(
        r"\b([\d¼½¾⅓⅔⅛⅜⅝⅞\/\.]+(?:\s+\d+\/\d+)?)\s*(tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons)\b",
        add_g, s, flags=re.IGNORECASE
    )

    # ===== POUND / LB / LBS =====
    def convert_lb(m):
        qty_raw = m.group(1)
        qty = safe_parse_quantity(qty_raw)
        if qty is None:
            return m.group(0)
        return f"{round(qty*453.592)}g"

    # (2-pound)
    s = re.sub(
        r"\(\s*([\d¼½¾⅓⅔⅛⅜⅝⅞\/\.]+(?:\s+\d+\/\d+)?)\s*[- ]*(pound|pounds|lb|lbs)\s*\)",
        lambda m: f"({convert_lb(m)})",
        s, flags=re.IGNORECASE
    )

    # 2-pound
    s = re.sub(
        r"\b([\d¼½¾⅓⅔⅛⅜⅝⅞\/\.]+(?:\s+\d+\/\d+)?)\s*[- ]*(pound|pounds|lb|lbs)\b",
        convert_lb,
        s, flags=re.IGNORECASE
    )

    # ===== FIX TRƯỜNG HỢP LỖI "(28 to )" =====
    s = re.sub(r"\(\s*(\d+)\s*to\s*\)", lambda m: f"({m.group(1)}g)", s)
    s = re.sub(r"\(\s*(\d+)\s*to\s*0g\s*\)", lambda m: f"({m.group(1)}g)", s)
    s = re.sub(r"\(\s*(\d+)\s*to\s*[^\d]*\)", lambda m: f"({m.group(1)}g)", s)

    # ===== CLEANUP: XÓA CÁC PATTERN TRÙNG LẶP (Xg) (Xg) → (Xg) =====
    # Xóa các pattern như (15g) (15g) → (15g)
    s = re.sub(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", r"(\1g)", s)
    # Xóa các pattern như (15g)(15g) → (15g) (không có khoảng trắng)
    s = re.sub(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", r"(\1g)", s)
    # Xóa các pattern trùng lặp nhiều lần
    while re.search(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", s):
        s = re.sub(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", r"(\1g)", s)

    return s


# ============================================================
#  🚀 APPLY + SAVE
# ============================================================

# 🔥 Đổi path tại đây
input_path = os.path.join(DATA_RECIPES, "all_recipes_normalized_time.csv") 
df = pd.read_csv(input_path, encoding="utf-8-sig")
output_path = input_path.replace(".csv", "_ing_norm.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"📌 Đã đọc file: {input_path} ({len(df):,} dòng)")

print("🧪 Chuẩn hoá cột ingredients...")
df["ingredients"] = df["ingredients"].apply(normalize_text)

if "instructions" in df.columns:
    print("🧪 Chuẩn hoá cột instructions...")
    df["instructions"] = df["instructions"].apply(normalize_text)

df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu file sau chuẩn hoá tại:\n👉 {output_path}")


📌 Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized_time.csv (40,684 dòng)
🧪 Chuẩn hoá cột ingredients...
🧪 Chuẩn hoá cột instructions...
✅ Đã lưu file sau chuẩn hoá tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized_time_ing_norm.csv


In [13]:
# ======================================================
# 🧹 CLEANUP: XÓA CÁC PATTERN TRÙNG LẶP (Xg) (Xg) → (Xg)
# ======================================================

import pandas as pd
import re

# 1️⃣ Đọc file đã bị lỗi
input_path = os.path.join(DATA_RECIPES, "full_data_ing.csv")
output_path = input_path.replace(".csv", "_clean.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"📌 Đã đọc file: {input_path} ({len(df):,} dòng)")

# 2️⃣ Hàm cleanup pattern trùng lặp
def cleanup_duplicate_grams(text):
    if pd.isna(text):
        return text
    s = str(text)
    
    # Xóa các pattern như (15g) (15g) → (15g)
    # Pattern 1: có khoảng trắng giữa
    s = re.sub(r"\(\s*(\d+)g\s*\)\s+\(\s*\1g\s*\)", r"(\1g)", s)
    # Pattern 2: không có khoảng trắng (15g)(15g)
    s = re.sub(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", r"(\1g)", s)
    # Pattern 3: lặp lại nhiều lần (15g) (15g) (15g) → (15g)
    while re.search(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", s):
        s = re.sub(r"\(\s*(\d+)g\s*\)\s*\(\s*\1g\s*\)", r"(\1g)", s)
    
    # Xóa các pattern như 1tablespoon (15g) (15g) → 1tablespoon (15g)
    s = re.sub(r"(\w+)\s*\(\s*(\d+)g\s*\)\s+\(\s*\2g\s*\)", r"\1 (\2g)", s)
    s = re.sub(r"(\w+)\s*\(\s*(\d+)g\s*\)\s*\(\s*\2g\s*\)", r"\1 (\2g)", s)
    
    return s

# 3️⃣ Áp dụng cleanup
print("🧹 Đang cleanup pattern trùng lặp...")
if "ingredients" in df.columns:
    df["ingredients"] = df["ingredients"].apply(cleanup_duplicate_grams)
    print("✅ Đã cleanup cột ingredients")

if "instructions" in df.columns:
    df["instructions"] = df["instructions"].apply(cleanup_duplicate_grams)
    print("✅ Đã cleanup cột instructions")

# 4️⃣ Lưu file
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Đã lưu file sau cleanup tại:\n👉 {output_path}")

# 5️⃣ Kiểm tra ví dụ
print("\n🔍 Kiểm tra 5 dòng đầu (ingredients):")
for idx, row in df.head(5).iterrows():
    if "ingredients" in row and pd.notna(row["ingredients"]):
        ing = str(row["ingredients"])[:200]
        if "(15g) (15g)" in ing or "(5g) (5g)" in ing:
            print(f"⚠️  Dòng {idx}: Vẫn còn trùng lặp!")
            print(f"   {ing[:100]}...")
        else:
            print(f"✅ Dòng {idx}: OK")


📌 Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_ing.csv (40,444 dòng)
🧹 Đang cleanup pattern trùng lặp...
✅ Đã cleanup cột ingredients
✅ Đã cleanup cột instructions

💾 Đã lưu file sau cleanup tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_ing_clean.csv

🔍 Kiểm tra 5 dòng đầu (ingredients):
✅ Dòng 0: OK
✅ Dòng 1: OK
✅ Dòng 2: OK
✅ Dòng 3: OK
✅ Dòng 4: OK


In [9]:
# ======================================================
# 🗑️ XÓA Asian, World, Fusion KHI NHIỀU CUISINES
# ➕ KHÔNG THÊM "vietnamese" BAO GIỜ
# ➕ Nếu CHỈ CÓ 1 cuisine = "vietnamese" → đổi thành "Vietnamese"
# ======================================================

import pandas as pd

# 1️⃣ Đọc file
input_path = os.path.join(DATA_RECIPES, "all_recipes_normalized_time_ing_norm.csv")
output_path = input_path.replace(".csv", "_1.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path} ({len(df)} dòng)")

if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong CSV!")


def clean_cuisine(cuisine_field):
    if pd.isna(cuisine_field) or str(cuisine_field).strip() == "":
        return cuisine_field

    cuisines = [c.strip() for c in str(cuisine_field).split(",") if c.strip()]

    # -------------------------
    # Trường hợp chỉ có 1 cuisine
    # -------------------------
    if len(cuisines) == 1:
        only = cuisines[0]

        # "vietnamese" -> "Vietnamese"
        if only.lower() == "vietnamese":
            return "Vietnamese"

        # Chỉ 1 cuisine khác → giữ nguyên, KHÔNG thêm gì
        return only

    # -------------------------
    # Nếu có nhiều cuisine → xoá Asian / World / Fusion
    # -------------------------
    remove_list = ["Asian", "World", "Fusion"]
    cuisines = [c for c in cuisines if c not in remove_list]

    # Nếu xoá sạch → fallback "World"
    if not cuisines:
        return "World"

    # -------------------------
    # Xoá trùng (case-insensitive)
    # -------------------------
    final = []
    seen = set()
    for c in cuisines:
        key = c.lower()
        if key not in seen:
            final.append(c)
            seen.add(key)

    return ", ".join(final)


# 3️⃣ Áp dụng
df["recipe_cuisine"] = df["recipe_cuisine"].apply(clean_cuisine)


# 4️⃣ Lưu kết quả
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Đã lưu file tại:\n👉 {output_path}")

# 5️⃣ Hiển thị ví dụ
print("\n🔍 VÍ DỤ:")
examples = df.head(10)
for idx, row in examples.iterrows():
    print(f"- {row['title'][:50]}...")
    print(f"  Cuisine: {row['recipe_cuisine']}")


✅ Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized_time_ing_norm.csv (40684 dòng)

💾 Đã lưu file tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\all_recipes_normalized_time_ing_norm_1.csv

🔍 VÍ DỤ:
-  
Caesar Scalloped Potatoes
...
  Cuisine: American
-  
Chili Crisp Marinated Cheese
...
  Cuisine: American
-  
No-Bake Millionaire’s Shortbread
...
  Cuisine: American
-  4-Ingredient Pepper Pizza Bites...
  Cuisine: American
-  Air Fryer Green Bean Fries ...
  Cuisine: American
-  Coffee Brandy Alexander 
...
  Cuisine: American
-  Copycat Cracker Barrel Fried Apples...
  Cuisine: American
-  Curry-Chutney Pinwheels...
  Cuisine: American
-  Date-Based Coffee Creamer...
  Cuisine: American
-  Elevated Appletini...
  Cuisine: American


In [10]:
# ======================================================
# 🔧 Chuẩn hoá đơn vị đo trong cột [ingredients] và [instructions]
# - ounce / oz  ➜  chuyển sang gram
# - fluid ounce / fl oz ➜ chuyển sang ml
# - tbsp / tsp  ➜ thêm số gram trong ngoặc (..g)
# ======================================================

import pandas as pd
import numpy as np
import re

# 1️⃣ Đọc file gốc
input_path = os.path.join(DATA_RECIPES, "full_data.csv")
output_path = input_path.replace(".csv", "_1.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path}")
print(f"📦 Số dòng: {len(df)} | Số cột: {len(df.columns)}")

# 2️⃣ Bảng quy đổi
conversion_table = {
    "ounce": 28.35, "ounces": 28.35, "oz": 28.35,
    "tbsp": 15, "tablespoon": 15, "tablespoons": 15,
    "tsp": 5, "teaspoon": 5, "teaspoons": 5,
}

# 3️⃣ Hàm chuẩn hoá đơn vị trong 1 đoạn text (áp dụng cho ingredients và instructions)
def normalize_units(text):
    if pd.isna(text):
        return text
    s = str(text)

    # ➤ (6- to 8-ounce) → (170–227g)
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?)\s*[-–]?\s*(?:to)?\s*(\d+(?:\.\d+)?)\s*[- ]*(?:ounce|ounces|oz)\s*\)",
        lambda m: f"({round(float(m.group(1))*28.35)}–{round(float(m.group(2))*28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ (12 fluid ounce) → (354.8ml)
    s = re.sub(
        r"\(\s*(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\s*\)",
        lambda m: f"({round(eval(m.group(1).replace(' ', '+'))*29.57,1)}ml)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 8 fluid ounces → 236.6ml
    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(fluid\s*ounces?|fl\.?\s*oz)\b",
        lambda m: f"{round(eval(m.group(1).replace(' ', '+'))*29.57,1)}ml",
        s,
        flags=re.IGNORECASE
    )

    # ➤ 1 20-ounce → 1 (567g)
    s = re.sub(
        r"(\b\d+(?:\.\d+)?\b)\s+(\d+(?:\.\d+)?)[- ]?(ounce|oz)\b",
        lambda m: f"{m.group(1)} ({round(float(m.group(2))*28.35)}g)",
        s,
        flags=re.IGNORECASE
    )

    # ➤ (3-ounce) hoặc 3-ounce hoặc 3 ounce → 85.1 g
    s = re.sub(
        r"\(?(\d+(?:\.\d+)?)[- ]?(?:ounce|ounces|oz)\)?",
        lambda m: f"{round(float(m.group(1))*28.35,1)} g",
        s,
        flags=re.IGNORECASE
    )

    # ➤ tbsp / tsp → thêm (..g)
    def add_g_parentheses(m):
        qty = m.group(1)
        unit = m.group(2).lower()
        grams = conversion_table[unit]
        return f"{qty}{unit} ({grams}g)"

    s = re.sub(
        r"\b(\d+(?:\.\d+)?|\d+\s*\d*\/\d*)\s*(tbsp|tablespoon|tablespoons|tsp|teaspoon|teaspoons)\b",
        add_g_parentheses,
        s,
        flags=re.IGNORECASE
    )

    return s


# 4️⃣ Áp dụng vào các cột nếu tồn tại
if "ingredients" in df.columns:
    df["ingredients"] = df["ingredients"].apply(normalize_units)
    print("✅ Đã chuẩn hoá đơn vị trong [ingredients]")

if "instructions" in df.columns or "introductions" in df.columns:
    col = "instructions" if "instructions" in df.columns else "introductions"
    df[col] = df[col].apply(normalize_units)
    print(f"✅ Đã chuẩn hoá đơn vị trong [{col}]")

# 5️⃣ Lưu file
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã xử lý xong và lưu tại:\n👉 {output_path}")


✅ Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data.csv
📦 Số dòng: 40684 | Số cột: 23
✅ Đã chuẩn hoá đơn vị trong [ingredients]
✅ Đã chuẩn hoá đơn vị trong [instructions]
✅ Đã xử lý xong và lưu tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_1.csv


In [11]:
import pandas as pd

# 📂 Đường dẫn file
input_path = os.path.join(DATA_RECIPES, "full_data_1.csv")
output_path = input_path.replace(".csv", ".csv")

# 1️⃣ Đọc file
df = pd.read_csv(input_path, encoding="utf-8-sig")

# 2️⃣ Kiểm tra cột có chứa cuisine
if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong file!")

# 3️⃣ Thay thế giá trị
df["recipe_cuisine"] = df["recipe_cuisine"].replace("All", "World")

# (Tuỳ chọn) Nếu có ô NaN thì cũng gán thành "World"
df["recipe_cuisine"] = df["recipe_cuisine"].fillna("World")

# 4️⃣ Ghi lại file mới
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Đã chuyển toàn bộ 'All' → 'World' và lưu tại:\n👉 {output_path}")

# 5️⃣ In kiểm tra nhanh
print("\n🔎 5 dòng đầu sau khi đổi:")
print(df["recipe_cuisine"].head())


✅ Đã chuyển toàn bộ 'All' → 'World' và lưu tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\full_data_1.csv

🔎 5 dòng đầu sau khi đổi:
0    American
1    American
2    American
3    American
4    American
Name: recipe_cuisine, dtype: object


In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(DATA_RECIPES, "full_data_ing.csv"), encoding="utf-8-sig")
print(len(df))

39742


In [ ]:
import pandas as pd

path = os.path.join(DATA_USERS, "users-survey.csv")
df = pd.read_csv(path)

# 🔍 In toàn bộ tên cột thật (có thể copy ra để dùng chính xác)
for i, c in enumerate(df.columns, 1):
    print(f"{i}. {repr(c)}")


1. 'Dấu thời gian'
2. 'Giới tính (Gender) '
3. '  Độ tuổi (Age)  '
4. '  Bạn có hạn chế chế độ ăn nào không? (Do you have any dietary restrictions?)  '
5. '  Bạn có dị ứng với thực phẩm nào không? (Do you have any food allergies?)  '
6. 'Bạn thường quan tâm đến loại bữa ăn nào? (Which type of meal do you usually care about?)'
7. 'Mức độ kỹ năng nấu ăn của bạn là gì? (Your cooking skill level)'
8. 'Bạn có bao nhiêu thời gian để nấu ăn trung bình? (How much time do you usually have to cook?)'
9. ' Bạn yêu thích món ăn của quốc gia nào? (Which country’s cuisine do you like?)'
10. 'Bạn thích những món ăn Việt Nam nào sau đây? (Which Vietnamese dishes do you like?)'
11. 'Bạn thích những món ăn Thái Lan nào sau đây? (Which Thai dishes do you like?)'
12. 'Bạn thích những món ăn Trung Quốc nào sau đây? (Which Chinese dishes do you like?)  '
13. 'Bạn thích những món ăn Pháp nào sau đây? (Which French dishes do you like?)'
14. 'Bạn thích những món ăn Ấn Độ nào sau đây? (Which Indian dishes do yo

In [ ]:
# 📘 Bước 1: Import thư viện
import pandas as pd

# 📘 Bước 2: Đọc file CSV
path = os.path.join(DATA_USERS, "users-survey.csv")
df = pd.read_csv(path)

# 📘 Bước 3: Kiểm tra tên cột chính xác
df.columns.tolist()
# 📘 Bước 4: Liệt kê toàn bộ nội dung trong cột
col_name = "Có món ăn nào bạn thích mà chưa được đề cập trong danh sách trên không? (nếu có)\nIs there any dish you like that was not mentioned in the list above?"
unique_answers = df[col_name].dropna().unique()

# 📘 Bước 5: Hiển thị kết quả
for i, val in enumerate(unique_answers, 1):
    print(f"{i}. {val}")



1. không
2. Có nhiều, liệt kê ko hết
3. Bún đậu mắm tôm
4. Panna cotta
5. Bì Bún (Việt Nam)
6. Cao lầu, bún bò Huế
7. No
8. Nhiều lắm
9. Bánh tráng trộn
10. Hong
11. Không
12. Ko có
13. Cơm sườn 
14. Bún đậu mắm tôm và takoyaky
15. Mì cayyy
16. Cơm Cuộn Hàn Quốc
17. Hog
18. Bún Bò
19. Không có
20. Gỏi cuốn
21. bánh tacos Pháp
22. Takoyaki
23. Tất cả các món Việt Nam
24. Lobster
25. không, món nào cũng thích 
26. Bột Masala của Ấn Độ
27. Nấm kim châm , nấm đùi gà và mì cay
28. Canh chua
29. bún bò
30. Cá viên chiên 
31. Coem Tấm
32. tôi thích tất cả các món ăn trên thế giới💗🥰
33. Mì cay
34. Masala là ngon nhất thế giới
35. Hamburger và phô mai 
36. không có
37. bánh khọt,bánh đúc nóng,quảng,súp don,ram bắp,cơm chiên dương châu,bột chiên
38. Bún bò, sườn xào chua ngọt, cơm sườn, joliebee 
39. cà ri ấn độ
40. Cơm cuộn tam giác (Hàn), Natto(Nhật),tacos(pháp),burito(mexico),Bún đậu( việt nam), sandwich (mỹ)
41. Bánh canh
42. mohammad salah, asalamulakum cà ra tu ni
43. ko
44. Cơm chiên dươn

In [2]:
import pandas as pd

# 📁 Đường dẫn input và output
file1 = os.path.join(DATA_RECIPES, "full_data.csv")
file2 = os.path.join(DATA_RECIPES, "combined_hungryhuy_vickypham_simulated_ratings_vn.csv")
output_path = os.path.join(DATA_RECIPES, "merged_full_data.csv")

# 📘 Đọc dữ liệu
df1 = pd.read_csv(file1, encoding="utf-8-sig")
df2 = pd.read_csv(file2, encoding="utf-8-sig") 

print("🔹 File 1:", df1.shape)
print("🔹 File 2:", df2.shape)

# 🧩 Thêm cột còn thiếu để đồng bộ
for col in df2.columns:
    if col not in df1.columns:
        print(f"🧩 Thêm cột {col} vào file1")
        df1[col] = None

for col in df1.columns:
    if col not in df2.columns:
        print(f"🧩 Thêm cột {col} vào file2")
        df2[col] = None

# 🔄 Sắp xếp lại cùng thứ tự cột
df2 = df2[df1.columns]

# 🧷 Gộp dữ liệu
combined = pd.concat([df1, df2], ignore_index=True)

# 🚫 Xoá trùng (nếu cùng title và url)
if "title" in combined.columns and "url" in combined.columns:
    before = combined.shape[0]
    combined.drop_duplicates(subset=["title", "url"], inplace=True)
    after = combined.shape[0]
    print(f"✅ Đã xoá {before - after} dòng trùng lặp")

# 💾 Lưu file hợp nhất
combined.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"🎉 File đã được gộp và lưu tại: {output_path}")
print("📦 Kích thước cuối cùng:", combined.shape)

🔹 File 1: (39742, 23)
🔹 File 2: (457, 24)
🧩 Thêm cột is_vn_dish vào file1
✅ Đã xoá 0 dòng trùng lặp
🎉 File đã được gộp và lưu tại: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\merged_full_data.csv
📦 Kích thước cuối cùng: (40199, 24)


In [16]:
# ======================================================
# 🔄 GỘP CUISINE THEO MAPPING
# Mediterranean → European
# Australian → World
# Fusion → World
# African → World
# Jewish → European
# ======================================================

import pandas as pd

# 1️⃣ Đọc file gốc (file mới nhất đã xử lý)
input_path = os.path.join(DATA_RECIPES, "merged_full_data_clean_normalized_time_ing_norm_1.csv")
output_path = input_path.replace(".csv", "_merged_cuisine.csv")

df = pd.read_csv(input_path, encoding="utf-8-sig")
print(f"✅ Đã đọc file: {input_path} ({len(df)} dòng)")

if "recipe_cuisine" not in df.columns:
    raise ValueError("❌ Không tìm thấy cột 'recipe_cuisine' trong CSV!")

# 2️⃣ Định nghĩa mapping
cuisine_mapping = {
    "Mediterranean": "European",
    "Australian": "World",
    "Fusion": "World",
    "African": "World",
    "Jewish": "European"
}

# 3️⃣ Hàm gộp cuisine
def merge_cuisine(cuisine_field):
    if pd.isna(cuisine_field) or str(cuisine_field).strip() == "":
        return "World"
    
    # Tách các cuisine (có thể có nhiều cuisine cách nhau bởi dấu phẩy)
    cuisines = [c.strip() for c in str(cuisine_field).split(",") if c.strip()]
    
    # Áp dụng mapping
    merged_cuisines = []
    for c in cuisines:
        if c in cuisine_mapping:
            merged_cuisines.append(cuisine_mapping[c])
        else:
            merged_cuisines.append(c)
    
    # Xoá trùng và giữ thứ tự
    seen = set()
    result = []
    for item in merged_cuisines:
        if item not in seen:
            seen.add(item)
            result.append(item)
    
    # Nếu sau khi gộp chỉ còn 1 cuisine, trả về string đơn
    if len(result) == 1:
        return result[0]
    
    # Nếu có nhiều cuisine, trả về dạng "Cuisine1, Cuisine2"
    return ", ".join(result)

# 4️⃣ Áp dụng mapping
print("\n🔄 Đang gộp cuisine...")
df["recipe_cuisine"] = df["recipe_cuisine"].apply(merge_cuisine)

# 5️⃣ Lưu file kết quả
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu file tại:\n👉 {output_path}")

# 6️⃣ Thống kê sau khi gộp
print("\n📊 Thống kê cuisine sau khi gộp:")
cuisine_counts = {}
for cell in df["recipe_cuisine"]:
    cuisines = [c.strip() for c in str(cell).split(",") if c.strip()]
    for c in cuisines:
        cuisine_counts[c] = cuisine_counts.get(c, 0) + 1

# Hiển thị TẤT CẢ các cuisine và số lượng
print("\n📈 TẤT CẢ CÁC CUISINE VÀ SỐ LƯỢNG:")
print("=" * 50)
sorted_cuisines = sorted(cuisine_counts.items(), key=lambda x: x[1], reverse=True)
for i, (c, cnt) in enumerate(sorted_cuisines, 1):
    print(f"{i:3}. {c:25} → {cnt:6}")

print("\n" + "=" * 50)
print(f"📊 Tổng số loại cuisine: {len(cuisine_counts)}")
print(f"📊 Tổng số recipe: {len(df)}")

# 7️⃣ Hiển thị ví dụ
print("\n🔍 VÍ DỤ 10 dòng đầu sau khi gộp:")
examples = df.head(10)
for idx, row in examples.iterrows():
    print(f"- {row['title'][:50]}...")
    print(f"  Cuisine: {row['recipe_cuisine']}")


C:\Users\ThaoMuy\AppData\Local\Temp\ipykernel_9324\3736921899.py:16: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path, encoding="utf-8-sig")


✅ Đã đọc file: C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\merged_full_data_clean_normalized_time_ing_norm_1.csv (40192 dòng)

🔄 Đang gộp cuisine...
💾 Đã lưu file tại:
👉 C:\Users\ThaoMuy\Downloads\Recommend_System\backend\data\recipes\merged_full_data_clean_normalized_time_ing_norm_1_merged_cuisine.csv

📊 Thống kê cuisine sau khi gộp:

📈 TẤT CẢ CÁC CUISINE VÀ SỐ LƯỢNG:
  1. American                  →  20072
  2. World                     →  11694
  3. Italian                   →   2235
  4. Mexican                   →   2033
  5. European                  →   1179
  6. Asian                     →    916
  7. French                    →    566
  8. British                   →    546
  9. Indian                    →    528
 10. Vietnamese                →    520
 11. Chinese                   →    367
 12. Caribbean                 →    344
 13. Thai                      →    231
 14. Japanese                  →    225
 15. Korean                    →    139

📊 Tổng